# Notebook 2 — Serve the Hugging Face AWQ model with vLLM and ngrok

**Target:** Google Colab T4 GPU. This notebook downloads the quantized repository created by Notebook 1, serves it through vLLM, tests the endpoint, and opens an HTTPS ngrok tunnel for your local machine.

## Compatibility strategy (checked 14 October 2026)

| Component | Pin / choice | Reason |
|---|---|---|
| Python | 3.12 in an isolated `uv` environment | Avoids mutating Colab's preinstalled Python/Torch stack |
| vLLM | `0.18.0` | Release pins `compressed-tensors==0.13.0` |
| PyTorch serving stack | `2.10.0` with CUDA 12.8 wheels | Matches the vLLM 0.18.0 release requirements and reported working version combination |
| Transformers / Hub | `4.57.3` / `0.36.2` | Keeps vLLM 0.18.0's Transformers `<5` requirement satisfied |
| Checkpoint | Hub repo from Notebook 1 | Must declare `quant_method: compressed-tensors` |
| Public API | vLLM OpenAI-compatible HTTP + API key + ngrok HTTPS | Local client and BentoML can call `/v1/chat/completions` |

**Do not install LLM Compressor in this environment.** vLLM `0.18.0` explicitly requires `compressed-tensors==0.13.0`; installing the quantizer and serving engine together can cause dependency resolution problems in other version combinations. The notebooks use separate Python environments and exchange only the uploaded model artifact. Sources: [vLLM 0.18.0 common requirements](https://github.com/vllm-project/vllm/blob/v0.18.0/requirements/common.txt), [CUDA requirements](https://github.com/vllm-project/vllm/blob/v0.18.0/requirements/cuda.txt), and [real-world version-conflict report](https://github.com/vllm-project/llm-compressor/issues/2258).

This pinned release is intentionally conservative, not the latest vLLM. Runtime checks and API tests are included, but the notebook has not been executed on a live Colab T4 from this chat.

## 1. Select and verify the Colab GPU

Use a fresh Colab runtime and choose **T4 GPU**. The first cell checks the actual GPU and CUDA availability.

In [1]:
import os, sys, subprocess, platform

print("Base Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi"], check=False)
import torch
print("Colab base Torch:", torch.__version__, "CUDA build:", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select Runtime > Change runtime type > T4 GPU, reconnect, and rerun.")
name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {name}; capability: {capability}; VRAM: {vram_gib:.1f} GiB")
if capability < (7, 5):
    raise RuntimeError(f"This workflow expects compute capability >= 7.5; found {capability}.")

Base Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
Colab base Torch: 2.11.0+cu130 CUDA build: 13.0
GPU: Tesla T4; capability: (7, 5); VRAM: 14.6 GiB


## 2. Configure the Hub repo and API credentials

Enter the model repo uploaded by Notebook 1. If it is private, provide a Hugging Face token with read access. You will also set an API key for vLLM: your local machine will need both the ngrok URL and this key.

In [2]:
from getpass import getpass

MODEL_ID = input("Hugging Face model repo from Notebook 1 (username/repo-name): ").strip()
if "/" not in MODEL_ID or MODEL_ID.startswith("YOUR_"):
    raise ValueError("Enter a real Hugging Face repo ID in username/repo-name format.")
HF_TOKEN = getpass("Hugging Face read token (hidden; blank if the repo is public): ").strip()
if HF_TOKEN and not HF_TOKEN.startswith("hf_"):
    raise ValueError("The token should begin with 'hf_', or leave it blank for a public repo.")
VLLM_API_KEY = getpass("Create a strong API key for the vLLM server (hidden; reuse locally): ").strip()
if len(VLLM_API_KEY) < 24:
    raise ValueError("Use an API key of at least 24 characters.")
SERVED_MODEL_NAME = "qwen2.5-3b-awq"
MAX_MODEL_LEN = 4096
print("Hub model:", MODEL_ID)
print("Served model name:", SERVED_MODEL_NAME)
print("Context length:", MAX_MODEL_LEN)
print("Hub token supplied:", bool(HF_TOKEN))

Hugging Face model repo from Notebook 1 (username/repo-name): abdelrahman-omran/qwen2.5-3b-instruct-awq-w4a16
Hugging Face read token (hidden; blank if the repo is public): ··········
Create a strong API key for the vLLM server (hidden; reuse locally): ··········
Hub model: abdelrahman-omran/qwen2.5-3b-instruct-awq-w4a16
Served model name: qwen2.5-3b-awq
Context length: 4096
Hub token supplied: True


## 3. Create an isolated vLLM environment

This uses a fresh environment and leaves Colab's base Torch installation untouched. It deliberately does **not** install `llmcompressor`.

In [3]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path("/content/qwen25-awq-vllm-serving")
ENV_PATH = ROOT / "venv"
SERVE_PY = ENV_PATH / "bin" / "python"
SERVE_CLI = ENV_PATH / "bin" / "vllm"
PYTHON_VERSION = "3.12"
VLLM_VERSION = "0.18.0"
ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", PYTHON_VERSION], check=True)
if not SERVE_PY.exists():
    subprocess.run(["uv", "venv", "--python", PYTHON_VERSION, "--managed-python", str(ENV_PATH)], check=True)
print("Serving Python:", SERVE_PY)

Serving Python: /content/qwen25-awq-vllm-serving/venv/bin/python


## 4. Install and verify the vLLM stack

The CUDA/PyTorch and `compressed-tensors` versions are checked after installation. Do not continue if `uv pip check` fails or the version assertions fail.

In [4]:
subprocess.run([
    "uv", "pip", "install", "--python", str(SERVE_PY), "--torch-backend=cu128",
    f"vllm=={VLLM_VERSION}",
    "torch==2.10.0", "torchaudio==2.10.0", "torchvision==0.25.0",
    "transformers==4.57.3", "huggingface-hub==0.36.2", "ninja==1.13.0",
], check=True)
subprocess.run(["uv", "pip", "check", "--python", str(SERVE_PY)], check=True)

# Verify Ninja was installed into this serving environment.
NINJA_EXE = ENV_PATH / "bin" / "ninja"
if not NINJA_EXE.exists():
    raise RuntimeError(f"Ninja executable not found at {NINJA_EXE}")
subprocess.run([str(NINJA_EXE), "--version"], check=True)

inspect_script = r"""import sys, torch
from importlib.metadata import version
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "CUDA build:", torch.version.cuda)
for package in ("vllm", "compressed-tensors", "transformers", "huggingface-hub"):
    print(f"{package}:", version(package))
assert version("vllm") == "0.18.0"
assert version("compressed-tensors") == "0.13.0"
assert version("transformers") == "4.57.3"
assert version("huggingface-hub") == "0.36.2"
assert version("torch").split("+")[0] == "2.10.0"
assert torch.cuda.is_available(), "Serving environment cannot see CUDA"
"""
subprocess.run([str(SERVE_PY), "-c", inspect_script], check=True)

CompletedProcess(args=['/content/qwen25-awq-vllm-serving/venv/bin/python', '-c', 'import sys, torch\nfrom importlib.metadata import version\nprint("Python:", sys.version.split()[0])\nprint("Torch:", torch.__version__, "CUDA build:", torch.version.cuda)\nfor package in ("vllm", "compressed-tensors", "transformers", "huggingface-hub"):\n    print(f"{package}:", version(package))\nassert version("vllm") == "0.18.0"\nassert version("compressed-tensors") == "0.13.0"\nassert version("transformers") == "4.57.3"\nassert version("huggingface-hub") == "0.36.2"\nassert version("torch").split("+")[0] == "2.10.0"\nassert torch.cuda.is_available(), "Serving environment cannot see CUDA"\n'], returncode=0)

## 5. Verify access and checkpoint metadata on the Hub

This fetches only `config.json` and checks the checkpoint metadata before vLLM loads the full weights.

In [5]:
import json

check_repo_script = r"""import json, os
from huggingface_hub import hf_hub_download
repo_id = os.environ["MODEL_ID"]
token = os.environ.get("HF_TOKEN") or None
path = hf_hub_download(repo_id=repo_id, filename="config.json", token=token)
config = json.load(open(path, encoding="utf-8"))
qconfig = config.get("quantization_config", {})
method = qconfig.get("quant_method", qconfig.get("quantization_method", "not found"))
print("Downloaded config:", path)
print("Quantization config:", qconfig)
print("Quantization method:", method)
if method not in ("compressed-tensors", "compressed_tensors"):
    raise RuntimeError(f"Expected compressed-tensors model, got {method!r}")
"""
check_env = os.environ.copy()
check_env["MODEL_ID"] = MODEL_ID
if HF_TOKEN:
    check_env["HF_TOKEN"] = HF_TOKEN
else:
    check_env.pop("HF_TOKEN", None)
subprocess.run([str(SERVE_PY), "-c", check_repo_script], env=check_env, check=True)

CompletedProcess(args=['/content/qwen25-awq-vllm-serving/venv/bin/python', '-c', 'import json, os\nfrom huggingface_hub import hf_hub_download\nrepo_id = os.environ["MODEL_ID"]\ntoken = os.environ.get("HF_TOKEN") or None\npath = hf_hub_download(repo_id=repo_id, filename="config.json", token=token)\nconfig = json.load(open(path, encoding="utf-8"))\nqconfig = config.get("quantization_config", {})\nmethod = qconfig.get("quant_method", qconfig.get("quantization_method", "not found"))\nprint("Downloaded config:", path)\nprint("Quantization config:", qconfig)\nprint("Quantization method:", method)\nif method not in ("compressed-tensors", "compressed_tensors"):\n    raise RuntimeError(f"Expected compressed-tensors model, got {method!r}")\n'], returncode=0)

## 6. Start vLLM and wait for its health endpoint

The API key protects inference calls. The T4 settings use a 4k context window, two concurrent sequences, and eager execution as conservative first-run choices. If startup runs out of memory, reduce the context length to 2048.

In [11]:
import getpass, subprocess, time, urllib.request, shutil

if "server_proc" in globals() and server_proc.poll() is None:
    raise RuntimeError("A vLLM server process is already running. Stop it before starting another one.")
SERVER_LOG = Path("/content/vllm-server.log")
server_args = [
    str(SERVE_CLI), "serve", MODEL_ID,
    "--host", "0.0.0.0",
    "--port", "8000",
    "--served-model-name", SERVED_MODEL_NAME,
    "--quantization", "compressed-tensors",
    "--dtype", "half",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", "0.88",
    "--max-num-seqs", "2",
    "--enforce-eager",
    "--api-key", VLLM_API_KEY,
]
server_env = os.environ.copy()
# uv virtualenv is not activated in the Colab notebook process.
# Put its bin directory on PATH so FlashInfer can find the Ninja executable.
server_env["PATH"] = str(ENV_PATH / "bin") + os.pathsep + server_env.get("PATH", "")
if shutil.which("ninja", path=server_env["PATH"]) is None:
    raise RuntimeError("Ninja is missing from the vLLM server PATH.")
if HF_TOKEN:
    server_env["HF_TOKEN"] = HF_TOKEN
else:
    server_env.pop("HF_TOKEN", None)
server_env["TOKENIZERS_PARALLELISM"] = "false"
server_log_handle = open(SERVER_LOG, "w")
server_proc = subprocess.Popen(server_args, stdout=server_log_handle, stderr=subprocess.STDOUT,
                               start_new_session=True, env=server_env)
print("Started vLLM PID:", server_proc.pid)
print("Server log:", SERVER_LOG)

ready = False
for attempt in range(120):
    if server_proc.poll() is not None:
        print(SERVER_LOG.read_text(errors="replace")[-14000:])
        raise RuntimeError(f"vLLM exited with code {server_proc.returncode}; see server log.")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(3)
if not ready:
    print(SERVER_LOG.read_text(errors="replace")[-14000:])
    raise TimeoutError("vLLM did not become healthy within six minutes.")
print("vLLM healthy at http://127.0.0.1:8000")

Started vLLM PID: 7239
Server log: /content/vllm-server.log
vLLM healthy at http://127.0.0.1:8000


## 7. Test the API inside Colab

In [12]:
import json, urllib.request

payload = {
    "model": SERVED_MODEL_NAME,
    "messages": [{"role": "user", "content": "Explain AWQ in one sentence."}],
    "max_tokens": 100,
    "stream": False,
}
request = urllib.request.Request(
    "http://127.0.0.1:8000/v1/chat/completions",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {VLLM_API_KEY}"},
    method="POST",
)
with urllib.request.urlopen(request, timeout=120) as response:
    result = json.loads(response.read().decode())
print(result["choices"][0]["message"]["content"])

AWQ stands for Automatic Question Generation, which is a technology that automatically generates questions based on given text or context.


## 8. Expose vLLM through ngrok HTTPS

Create an ngrok account and copy your authtoken from its dashboard. The URL and vLLM API key together grant access to your live Colab model. Do not publish either. Keep this notebook runtime and the tunnel alive while using the local client.

In [13]:
import subprocess, sys
from getpass import getpass
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
from pyngrok import ngrok

if "ngrok_tunnel" in globals():
    try:
        ngrok.disconnect(ngrok_tunnel.public_url)
    except Exception:
        pass
NGROK_TOKEN = getpass("ngrok authtoken (hidden): ").strip()
if not NGROK_TOKEN:
    raise ValueError("An ngrok authtoken is required.")
ngrok.set_auth_token(NGROK_TOKEN)
ngrok_tunnel = ngrok.connect(addr=8000, proto="http", bind_tls=True)
PUBLIC_URL = ngrok_tunnel.public_url
if not PUBLIC_URL.startswith("https://"):
    raise RuntimeError(f"Expected HTTPS tunnel URL, got {PUBLIC_URL}")
print("Public base URL:", PUBLIC_URL)
print("OpenAI-compatible base URL:", PUBLIC_URL + "/v1")
print("Served model name:", SERVED_MODEL_NAME)
print("Copy the HTTPS URL and the API key you entered above to your local machine; do not share them.")

ngrok authtoken (hidden): ··········
Public base URL: https://b560-34-148-29-126.ngrok-free.app
OpenAI-compatible base URL: https://b560-34-148-29-126.ngrok-free.app/v1
Served model name: qwen2.5-3b-awq
Copy the HTTPS URL and the API key you entered above to your local machine; do not share them.


In [14]:
import urllib.request

with urllib.request.urlopen(
    "http://127.0.0.1:8000/health", timeout=5
) as response:
    print("vLLM health:", response.status)

vLLM health: 200


## 9. Connect your local Ubuntu machine to this server

### First, patch the existing backend to send the API key

Your current `src/legal_rag/core/llm_backends.py` uses `OpenAI(api_key="not-needed", base_url=base_url)`. Since this vLLM server is protected by an API key, change `LlamaServerBackend.__init__` to:

```python
self.client = OpenAI(
    api_key=os.getenv("LLM_API_KEY", "not-needed"),
    base_url=base_url,
    default_headers={"ngrok-skip-browser-warning": "true"},
)
```

The module already imports `os`. Keep `LLM_BACKEND=llamacpp` for now because your current backend factory supports `transformers|llamacpp`, and the `llamacpp` implementation already calls the OpenAI-compatible `/chat/completions` endpoint. We are reusing that client for vLLM; you do **not** need to run llama.cpp locally for this session.

### Then set the environment and start BentoML

Copy the actual ngrok URL and the same API key entered in this notebook. Run these commands from your local project checkout, in the same terminal session:

```bash
# Keep Qdrant running locally, but the local llama.cpp container is no longer needed.
docker compose stop llm

export VLLM_BASE_URL="https://YOUR-NGROK-HOST.ngrok-free.app/v1"
export VLLM_API_KEY="PASTE_THE_SAME_VLLM_API_KEY"

LLM_BACKEND=llamacpp \
LLM_BASE_URL="$VLLM_BASE_URL" \
LLM_API_KEY="$VLLM_API_KEY" \
LLM_MODEL_NAME="qwen2.5-3b-awq" \
QDRANT_URL="http://localhost:6333" \
bentoml serve legal_rag.serving.bentoml_service:LegalRAGService
```

**Important:** `LLM_MODEL_NAME` must match the `--served-model-name` value in this notebook. In this setup, BentoML, your retrieval code, and Qdrant remain local; only model generation goes through the encrypted ngrok URL to the Colab T4.

## Troubleshooting and safety notes

- **vLLM fails to start:** inspect `/content/vllm-server.log`; run `nvidia-smi`; verify `compressed-tensors==0.13.0`; keep `--enforce-eager`; try `--max-model-len 2048`.
- **A version assertion or `uv pip check` fails:** stop instead of force-reinstalling packages. Do not install LLM Compressor into the serving environment.
- **Private Hub repo returns 401/403:** provide a read-enabled token; confirm the repo ID is correct.
- **vLLM returns 401 locally:** verify `LLM_API_KEY` exactly matches the API key entered in this notebook.
- **Model name error:** use exactly `qwen2.5-3b-awq` in `LLM_MODEL_NAME`, or change both the server’s `--served-model-name` and local setting.
- **ngrok 502/timeouts:** the Colab runtime, vLLM process and tunnel must all be live. Tunnel URLs can change when reconnecting.
- **T4 kernel/runtime reports:** user reports show some T4-specific vLLM/Triton failures for other workloads, so the first server health check and a real chat completion are necessary rather than assuming install success proves inference works ([example report](https://github.com/vllm-project/vllm/issues/20259)).
- **Security:** ngrok exposes a public URL. Keep the API key private, stop the tunnel when finished, and do not treat a free Colab session as a production service.

## Stop vLLM and disconnect ngrok when finished

In [15]:
try:
    if "ngrok_tunnel" in globals():
        ngrok.disconnect(ngrok_tunnel.public_url)
        print("ngrok tunnel disconnected")
except Exception as exc:
    print("ngrok disconnect warning:", exc)

try:
    if "server_proc" in globals() and server_proc.poll() is None:
        server_proc.terminate()
        try:
            server_proc.wait(timeout=15)
        except subprocess.TimeoutExpired:
            server_proc.kill()
        print("vLLM server stopped")
except Exception as exc:
    print("vLLM stop warning:", exc)

ngrok tunnel disconnected
vLLM server stopped
